# 02 - İş Yükü Analizi

## Amaç ve Kapsam

Bu notebook, tezin donanım tahsisi ve derin öğrenme (Deep Learning) modelleme süreçlerine yön veren temel dinamikleri ampirik biçimde ortaya koyan kapsamlı bir **iş yükü karakterizasyonunu (workload characterization)** temsil etmektedir. Sonraki bölümlerde (Notebook 04 ve 05) inşa edilecek olan Random Forest, XGBoost, LightGBM standart modelleri ile sıralı (sequential) belleğe sahip derin öğrenme (CNN, LSTM, Hibrit) modellerinin başarısı, tamamen burada deşifre edilen veri örüntülerine dayanmaktadır.

Bu aşamada elde edilen istatistiksel kanıtlar, tez mimarisini doğrudan şekillendirmek için yapılandırılmıştır:

1. **Çalışma Süresi (Runtime) Varyasyonları:** İş sürelerinin ağır kuyruklu (heavy-tail) doğasını kanıtlayarak ortalama tabanlı naif hata metriklerinin (MSE) uygunsuzluğunu bilimsel olarak temellendirmek ve MAE/MdAE kullanımını zorunlu kılmak.
2. **Kuyruk Tıkanıklıkları ve HoL (Head-Of-Line) Riski:** Geleneksel FIFO politikalarının küme (cluster) üzerinde yaratacağı yıkıcı darboğaz potansiyelini istatistiksel varyasyonlarla ispatlamak ve makine öğrenmesi destekli adaptif (SJF + ML) tahmine duyulan hayati ihtiyacı vurgulamak.
3. **Zamansal Sıralı (Sequential) Gelişler:** İş yükündeki ani sıçramaların (bursts), saatlik (diurnal) yoğunlukların ve süreksiz aralıkların (inter-arrival times), neden LSTM gibi zaman bağımlı sinir ağları ile analiz edilmesi gerektiğini haklı çıkarmak.
4. **Gerçekçi Simülasyon Gereksinimi:** Parçalı (fragmented) ve değişken GPU kısıtlarının (1'den 8 GPU'ya kadar), neden tek-kuyruklu denklem modelleri yerine doğrudan özel yapım `MultiNodeClusterSimulator` aracı üzerinde test edilmesi gerektiğini doğrulamak.

Bu analiz doğrudan veriden hareketle bir savunma çizgisi oluşturur; teorik yaklaşımlardan ziyade verinin bizzat talep ettiği ağaç tabanlı ve derin öğrenme tabanlı mimarilerin inşasına nesnel bir geçiş köprüsü kurar.


In [1]:
# ── 0. Environment & Path Setup ──────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Anchor on a marker that only exists at the repository root, so the notebook
# works regardless of the directory the kernel was started from. parents[1]
# resolved one level short when the working directory was notebooks/.
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    return start.parents[1]

PROJECT_ROOT = _find_project_root(Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[Setup] Project root : {PROJECT_ROOT}")
print(f"[Setup] Python path  : {sys.executable}")

[Setup] Project root : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling
[Setup] Python path  : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling/venv/bin/python


> **Kurulum doğrulandı.** `src.*` modülleri Python yoluna eklenmiş,
> `configs/paths.yaml` üzerinden dizinler doğrulanmıştır.

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import (
    load_prepared_job_table,
    compute_basic_stats,
    compute_runtime_histogram,
    compute_arrival_rate_series,
    summarize_workload,
)

# Unified visualisation theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.family": "DejaVu Sans",
    # ── Publication-grade output ───────────────────────────────────────────
    # scripts/export_thesis_results.py scrapes the notebook's own inline PNG,
    # so the inline dpi is what reaches the thesis. At the 100 dpi default
    # these exported at 147-280 ppi, under the 300 ppi publisher floor.
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    # Figures are authored on a generous canvas, 10 to 18 in, and printed into
    # a 6.10-in column, roughly a 0.35 to 0.55x reduction. Shrinking the canvas
    # to match the print width made every panel cramped, so the canvas stays
    # and the type is scaled instead: at 15 pt a 14-in figure prints at 6.5 pt,
    # above the ~6 pt legibility floor.
    "font.size": 15,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 19,
})

print("[Setup] All imports OK.")


> **Bağımlılıklar hazır.** NumPy, Pandas, Matplotlib, Seaborn ve tüm
> `src.analysis` modülleri başarıyla içe aktarılmıştır.

Veri ön işleme ve filtreleme adımlarının ardından elde edilen analiz veri seti **82.184 iş** ve **14 öznitelik** içermektedir. Bu özniteliklerin bir kısmı doğrudan ham kayıtlardan alınırken, analiz yapılabilmesi için bir kısmı sonradan türetilmiştir:

**Temel Özellikler:**
- `job_id`, 
- `gpu_demand`, 
- `user`, 
- `gpu_type`, 
- `num_inst`, 
- `num_cpu`, 
- `job_runtime` (Hedef Değişken)

**Zamansal Özellikler (Türetilmiş):**
İş yükünün zamana bağlı dağılımını görebilmek için ham zaman damgasından (timestamp) türetilmiştir:
- `arrival_time`, 
- `arrival_sec`, 
- `hour_of_day`, 
- `day_of_week`

**Sistem Yükü Özellikleri (Sweep-Line Algoritması):**
İşin geldiği andaki küme doluluğu doğrudan CPU/GPU kullanımından ziyade geliş-gidiş loglarının Sweep-Line Algoritması ile taranarak hesaplanmıştır:
- `cluster_load_cpu`, 
- `cluster_load_gpu`, 
- `active_job_count`

In [3]:
# ── 2. Load dataset ───────────────────────────────────────────────────────────
print("[Step 1] Loading and preparing job table...")
job_df = load_prepared_job_table(dataset="main", time_unit="s")

print(f"[Step 1] Jobs : {len(job_df):,}  |  Columns : {list(job_df.columns)}")
job_df.head(3)

[Step 1] Loading and preparing job table...
[Step 1] Jobs : 82,184  |  Columns : ['job_id', 'arrival_time', 'arrival_sec', 'job_runtime', 'gpu_demand', 'user', 'gpu_type', 'num_inst', 'num_cpu', 'hour_of_day', 'day_of_week', 'cluster_load_cpu', 'cluster_load_gpu', 'active_job_count']


,job_id,arrival_time,arrival_sec,job_runtime,gpu_demand,user,gpu_type,num_inst,num_cpu,hour_of_day,day_of_week,cluster_load_cpu,cluster_load_gpu,active_job_count
0,1,1970-01-01 00:00:03,3.0,15748.0,0.25,d4d51aca8806,T4,12.0,6.0,0,0,2.0,0.00,1
1,2,1970-01-01 00:00:05,5.0,84.0,1.00,a8192d6b0ae9,MISC,1.0,6.0,0,0,8.0,0.25,2
2,3,1970-01-01 00:00:21,21.0,46.0,1.00,c7152ce0fec1,T4,1.0,18.0,0,0,20.0,2.25,4


> Veri ön işleme ve filtreleme adımlarının ardından elde edilen veri seti **82.184 iş**
>ve toplam **14 adet öznitelik** içermektedir.
>
>**Temel Özellikler:**
>- `job_id` — İş tanımlayıcısı
>- `arrival_time` — Ham geliş zaman damgası
>- `arrival_sec` — İz başlangıcına göre saniye cinsinden geliş zamanı
>- `job_runtime` — Gerçek çalışma süresi (hedef değişken)
>- `gpu_demand` — Talep edilen GPU sayısı
>- `user` — İş sahibi kullanıcı (kategorik)
>- `gpu_type` — Kullanılan GPU tipi (kategorik)
>- `num_inst` — Kullanılan instance sayısı
>- `num_cpu` — Talep edilen CPU sayısı
>
>**Zamansal Özellikler:**
>- `hour_of_day` — Günün saati (0–23)
>- `day_of_week` — İz-göreli gün indeksi (0, 1, 2, ...; DOĞRULANMIŞ bir takvim günü DEĞİL -- izin genel yayınında toplama tarihi açıklanmamış)
>
>**Sistem Türevli Özellikler (Sweep-line):**
>- `cluster_load_cpu` — İşin geldiği andaki arka plan CPU yükü
>- `cluster_load_gpu` — İşin geldiği andaki arka plan GPU yükü
>- `active_job_count` — İşin geldiği anda kümede aktif iş sayısı

In [4]:
# ── 3. Scalar statistics ──────────────────────────────────────────────────────
stats = compute_basic_stats(job_df)

print("=" * 50)
print("  Workload Basic Statistics")
print("=" * 50)
for k, v in stats.items():
    print(f"  {k:<25}: {v:>12,.2f}")
print("=" * 50)

# One-row summary table (LaTeX-ready)
summary_df = summarize_workload(job_df)
display(summary_df)

  Workload Basic Statistics
  num_jobs                 :    82,184.00
  mean_runtime             :     5,223.02
  median_runtime           :       594.00
  p95_runtime              :    24,110.05
  p99_runtime              :    67,432.38
  mean_gpu_demand          :         0.68
  max_gpu_demand           :         8.00
  time_span_hours          :       183.86


,num_jobs,mean_runtime_sec,median_runtime_sec,p95_runtime_sec,p99_runtime_sec,mean_gpu_demand,max_gpu_demand,time_span_hours
0,82184,5223.019408,594.0,24110.05,67432.38,0.680211,8.0,183.858056


> İş yükünün zaman ve kaynak açısından yüksek derecede heterojen (değişken) olduğu görülmektedir. Ortalama iş süresi yaklaşık **5.223 saniye (≈ 1.5 saat)** iken, medyan sürenin yalnızca **594 saniye (≈ 10 dakika)** olması, iş sürelerinin **ağır kuyruklu (heavy-tailed)** bir dağılıma sahip olduğunu göstermektedir.
>
> Bu durum, işlerin büyük çoğunluğunun kısa sürede tamamlandığını; ancak az sayıdaki çok uzun işin ortalama süreyi önemli ölçüde yukarı çektiğini ortaya koymaktadır. Üst yüzdelik dilimlerde (percentile), iş sürelerinin P95 seviyesinde **~6.7 saat (24.110 s)** ve P99 seviyesinde **~18.7 saat (67.432 s)** mertebesine çıkması, nadir fakat devasa boyutlardaki işlerin sistem davranışında belirleyici bir etkiye sahip olduğunu gösterir. Bu muazzam varyasyon (CV >> 1), geleneksel normal dağılım varsayımını tümüyle geçersiz kılarak modellemede gürbüz (robust) hata metriklerinin (örn. MAE, MdAE) kullanımını zorunlu tutmaktadır.
>
> GPU talebi açısından incelendiğinde, ortalama GPU isteği 0.68 düzeyinde (yukarıda yazdırılan `mean_gpu_demand` satırı) seyretse de maksimum talebin aynı anda **8 GPU'ya** kadar çıkması, iş yükünde devasa bir parçalı kaynak (fragmentation) ihtiyacı bulunduğunu kanıtlamaktadır. Test edilen iz verisi (trace), 183.86 saatlik (~7.7 gün) kesintisiz bir zaman dilimine yayılmış **82.184 onaylanmış işi** kapsamaktadır.


In [ ]:
# ── 4. Runtime histogram ──────────────────────────────────────────────────────
hist_counts, bin_edges = compute_runtime_histogram(job_df, bins=60, log_scale=True)
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(bin_centres, hist_counts, width=np.diff(bin_edges), color="steelblue",
       edgecolor="white", linewidth=0.4)
ax.set_xlabel("log₁₀(Runtime + 1)")
ax.set_ylabel("Frequency")
ax.set_title("Job Runtime Distribution (Log₁₀ Scale)")
plt.tight_layout()
plt.show()

> **Şekil 1 — Çalışma Süresi (Runtime) Dağılımı (Log₁₀ Ölçek):**
>
> Kümedeki GPU iş sürelerinin logaritmik ölçekteki dağılımı incelendiğinde, iş yükünün doğası gereği şiddetli bir **bimodal (iki tepeli)** karakteristik sergilediği görülmektedir. İlk yoğunlaşma noktası log₁₀ ≈ 2.4 (~250 saniye) civarında kümelenmiş olup, bu durum sistemdeki on binlerce kısa ömürlü **çıkarım (inference)** veya hızlı test görevini temsil etmektedir. İkinci ve daha geniş tabanlı yoğunlaşma ise log₁₀ ≈ 3.6–3.7 (~4,000–5,000 saniye / ~1.1–1.4 saat) bandında belirginleşip yaklaşık log₁₀ ≈ 3.9'a kadar yüksek seyrederek uzun soluklu **model eğitimi (training)** veya devasa veri ön işleme işlerine işaret etmektedir.
>
> En kritik bulgu; logaritmik dönüşüme (log transform) rağmen dağılımın sağ kuyruğunun log₁₀ ≈ 5 (~100,000 saniye) seviyelerine kadar uzanması ve simetrik bir çan eğrisi (normal dağılım) oluşturmamış olmasıdır. Bu asimetrik yapı, makine öğrenmesi modellerinin eğitiminde (Bölüm 04) ortalamaya dayalı hataları (MSE, RMSE) şiddetle saptıracaktır; dolayısıyla model performansını ölçerken uç değerlere (outliers) karşı dirençli olan **MAE (Ortalama Mutlak Hata)** ve özellikle **MdAE (Medyan Mutlak Hata)** gibi regresyon metriklerinin kullanılması teorik bir mecburiyettir.

In [ ]:
# ── 5. Hourly arrival rate ────────────────────────────────────────────────────
# Note: uses lowercase "1h" alias — "1H" deprecated in pandas >= 2.2
arrival_series = compute_arrival_rate_series(job_df, freq="1h")

# Convert timedelta index to fractional days for readability
elapsed_days = arrival_series.index.total_seconds() / 86400

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(elapsed_days, arrival_series.values, lw=1.2, color="seagreen")
ax.fill_between(elapsed_days, arrival_series.values, alpha=0.15, color="seagreen")
ax.set_xlabel("Elapsed Time (days)")
ax.set_ylabel("Jobs per Hour")
ax.set_title("Hourly Job Arrival Rate Over Trace Duration")
plt.tight_layout()
plt.show()

> **Şekil 2 — Saatlik İş Geliş Hızı (Hourly Arrival Rate) (8 Günlük Kesit):**
>
> Küme üzerindeki trafik yoğunluğunun 8 günlük (yaklaşık 183 saat) zamansal evrimi izlendiğinde, standart kuyruk teorisinde sıklıkla varsayılan sabit hızlı (homojen Poisson) iş geliş modelinin Alibaba PAI kümesi için tamamen yetersiz kaldığı ispatlanmıştır. Sistemin olağan taban yükü saatte ortalama 200–400 iş arasında seyrederken; örneğin periyodun 1.5. gününde ani bir patlamayla **saatte ~1.350 işe**, 5.7. gününde ise **saatte ~1.080 işe** fırladığı gözlemlenmiştir.
>
> Zaman serisindeki bu şiddetli **sıçramalı (bursty)** yapı, anlık donanım darboğazlarının (hardware starvation) ana tetikleyicisidir. Sistemin bu dalgalanmalara göğüs gerebilmesi için, gelen statik iş yükünü pasif bir şekilde eriten basit FIFO'lardan öte, yaklaşan yük kümelerini öngörebilen proaktif ve makine öğrenmesi algılarına sahip çizelgeleme modellerine (SJF + Predictor) hayati ihtiyaç duyduğu açıkça görülmektedir. Zaman ekseni iz-göreli (trace-relative) bir eksendir: `submit_time` izin başlangıcında 0'dan başladığı için `arrival_time`, kümenin gerçek saatine değil Unix epoch'una sabitlenmiştir. Yukarıda çizilen geçen-gün ekseni bu nedenle kesindir; ancak üzerindeki hiçbir nokta bir takvim tarihine ya da duvar saati saatine karşılık gelmez (bkz. Notebook 01, Şekil 3 ve 6 notları).


In [ ]:
# ── 6. GPU demand vs runtime scatter ─────────────────────────────────────────
# All jobs, not a 5,000-job sample. The sample existed only to control
# overplotting, which jitter and low alpha already handle, and it dropped the
# tail this thesis is about: 397,234 s against 599,445 s in the full data.
sample = job_df

fig, ax = plt.subplots(figsize=(10, 6))
# The colour bar encoded gpu_demand, already on the x axis, and was drawn at
# full saturation while the points render at alpha 0.3, so the key matched no
# mark on the plot. Most requests are fractional, median 0.5, so a linear x
# axis squeezed every sub-1-GPU level into one smear at the origin; log x
# separates the sharing levels. Jitter and small transparent marks recover the
# within-column density.
_jit = np.random.default_rng(42).normal(0, 0.012, len(sample))
ax.scatter(sample["gpu_demand"] * (1 + _jit), sample["job_runtime"],
           alpha=0.035, s=4, color="#4C72B0", edgecolor="none",
           label=f"{len(sample):,} jobs")
# One-entry legend, but it carries the sample size, which is the one thing a
# reader cannot recover from the plot itself.
_leg = ax.legend(loc="upper left", markerscale=4, framealpha=0.9,
                 handletextpad=0.4, borderpad=0.5)
# The points are drawn at alpha=0.035 to show density; at that opacity the
# legend swatch is invisible, so the key marker is drawn opaque.
for _h in _leg.legend_handles:
    _h.set_alpha(0.9)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("GPU Demand per Job (GPUs, log scale)")
ax.set_ylabel("Runtime (seconds, log scale)")
ax.set_title("GPU Demand vs Job Runtime")
plt.tight_layout()
plt.show()

> **Şekil 3 — GPU Talebi vs. Çalışma Süresi (GPU Demand vs. Runtime Scatter - 82.184 İşin Tamamı):**
>
> Tablodaki işlerin tamamının, 5.000 işlik bir alt örneklemin değil, çift logaritmik ölçekte dağılımı incelendiğinde, talep edilen GPU miktarı ile elde edilen bitiş süresi arasında basit bir "doğrusal ilişki" (pozitif veya negatif korelasyon) bulunmadığı somutlaşmıştır. Yoğunluk, GPU paylaşımı seviyelerinde ve tam olarak **1 GPU** talep eden görevlerde toplanmaktadır; grafikte 0 GPU sütunu yoktur, çünkü tablo yalnızca GPU işlerini içermektedir (en düşük talep 0.01, medyan 0.5). Talepler 2 GPU'dan 8 GPU'ya doğru ilerledikçe nokta yoğunluğu keskin bir düşüş sergilemektedir; ancak bunlar tekil olaylar değildir: 6 GPU talep eden 21, 8 GPU talep eden 82 iş bulunmakta, medyan süreleri 2.8 × 10⁴ saniye civarında, maksimumları ise 4 × 10⁵ saniyenin üzerinde seyretmektedir.
>
> Bu görselin en güçlü ampirik mesajı şudur: **Sadece `gpu_demand` sütununa bakılarak bir işin ne kadar süreceği asla tahmin edilemez.** 1 GPU talep eden bir iş 6 saniye de sürebilmekte, 598.415 saniye (yaklaşık 6.9 gün) de devam edebilmektedir. Bu muazzam belirsizlik; yapay zeka modellerine `user` (kullanıcı kodlama alışkanlıkları), `gpu_type` (donanım mimarisi) ve `hour_of_day` gibi bağlamsal/zamansal özniteliklerin (Feature Engineering - Bölüm 03) dahil edilmesinin neden sadece bir performans iyileştirmesi değil, mutlak bir mimari zorunluluk olduğunu görsel bağlamda kanıtlamaktadır.


## Özet

Bu notebook'ta gerçekleştirilen iş yükü karakterizasyon analizi, çizelgeleme optimizasyonunun gerekliliğini ve önerilen ML tabanlı yaklaşımın geçerliliğini doğrudan ampirik verilerle pekiştirmiştir:

| Gözlem | Çizelgeleme Üzerindeki Etkisi |
|---|---|
| **Bursty (Sıçramalı) Geliş Süreci** | Klasik M/M/1 modellerini geçersiz kılar; adaptif çizelgeleme şarttır. |
| **Ağır-Kuyruk (Heavy-Tail) Süre Dağılımı** | Mean değerine dayalı FIFO, yıkıcı Head-Of-Line (HoL) bloklanmasına neden olur. |
| **Gündüz/Gece Döngüsü** | Zaman tabanlı özelliklerin (`hour_of_day`) tahmin için neden kritik olduğunu açıklar. |
| **Heterojen Kaynak Talepleri** | Basit kuyruk modelleri yerine düğüm tabanlı (per-node) simülatör kullanımını zorunlu kılar. |

Bu bulgular, tezin SJF + ML tahmin entegrasyonuna dayalı mimari seçimini yalnızca teorik seviyede değil, tamamen veriye dayalı (data-driven) argümanlarla desteklemektedir.